In [5]:

from pathlib import Path
from collections import Counter
import pandas as pd
import json, re, zipfile

PROJECT = Path("/kaggle/working/nd_iris_project")
SOURCE = Path("/kaggle/input/datasets/pranavsunilwagh/nd-iris/NDCLD15")
OUT = PROJECT / "notebook04_baseline_recognition"
OUT.mkdir(parents=True, exist_ok=True)

files = sorted(SOURCE.rglob("*.tiff"))
assert files, f"No TIFF images found in {SOURCE}"

rows = []
for p in files:
    stem = p.stem
    rows.append({
        "filename": p.name,
        "stem": stem,
        "parent_folder": str(p.parent.relative_to(SOURCE)),
        "prefix_before_first_letter": (
            re.split(r"[A-Za-z]", stem, maxsplit=1)[0] or None
        ),
        "leading_digits": (
            re.match(r"^\d+", stem).group(0)
            if re.match(r"^\d+", stem) else None
        ),
        "length": len(stem),
        "pattern": re.sub(r"\d+", "#", stem)
    })

df = pd.DataFrame(rows)
df.to_csv(OUT / "ND_Iris_filename_audit.csv", index=False)

print("=" * 65)
print("ND IRIS — IDENTITY LABEL AUDIT")
print("=" * 65)
print(f"TIFF files: {len(df):,}")
print(f"Unique filename stems: {df['stem'].nunique():,}")
print(f"Unique parent folders: {df['parent_folder'].nunique():,}")

print("\nMost common filename patterns:")
print(df["pattern"].value_counts().head(15).to_string())

print("\nMost common leading-digit groups (candidate only):")
print(df["leading_digits"].value_counts().head(20).to_string())

print("\nSample filenames:")
for name in df["filename"].head(30):
    print(" ", name)

print("\nFolder distribution:")
print(df["parent_folder"].value_counts().head(20).to_string())

candidate_counts = df["leading_digits"].value_counts(dropna=True)
summary = {
    "image_count": int(len(df)),
    "unique_stems": int(df["stem"].nunique()),
    "unique_parent_folders": int(df["parent_folder"].nunique()),
    "filename_patterns": {
        str(k): int(v) for k, v in df["pattern"].value_counts().items()
    },
    "leading_digit_group_count": int(df["leading_digits"].nunique(dropna=True)),
    "leading_digit_group_sizes": {
        str(k): int(v) for k, v in candidate_counts.items()
    },
    "identity_labels_verified": False,
    "conclusion": (
        "Filename patterns are exploratory candidates only. "
        "Do not train recognition until the subject/eye identity mapping "
        "is confirmed from dataset documentation or authoritative metadata."
    )
}
with open(OUT / "ND_Iris_identity_label_audit.json", "w") as f:
    json.dump(summary, f, indent=2)

conclusion = """## Conclusion — Notebook 04: Baseline Recognition Preparation

The ND Iris image filenames and directory structure were inspected to identify
possible identity-label conventions. Filename-derived groups are exploratory
and have not been verified as subject or eye identities.

A baseline recognition model must not be trained with unverified labels, because
the resulting accuracy would not have a reliable biometric interpretation.
The identity mapping and appropriate subject-disjoint split must be established
before training and evaluation.
"""
(OUT / "Notebook04_Conclusion.md").write_text(conclusion)

zip_path = PROJECT / "ND_Iris_Notebook04_Results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook04_baseline_recognition/{p.name}")

with zipfile.ZipFile(zip_path) as z:
    bad = z.testzip()

print("\nAudit CSV:", OUT / "ND_Iris_filename_audit.csv")
print("Conclusion Markdown:", OUT / "Notebook04_Conclusion.md")
print("Results ZIP:", zip_path)
print("ZIP integrity:", "PASSED" if bad is None else f"FAILED: {bad}")
assert bad is None


ND IRIS — IDENTITY LABEL AUDIT
TIFF files: 6,233
Unique filename stems: 6,233
Unique parent folders: 1

Most common filename patterns:
pattern
#d#       6219
#d#(#)      14

Most common leading-digit groups (candidate only):
leading_digits
07013    761
07014    519
06345    153
06312    149
06017     30
07049     20
07033     20
06883     20
06886     20
07040     20
07042     20
07026     20
07029     20
07048     20
06903     20
06906     20
06915     20
06907     20
06888     20
06895     20

Sample filenames:
  04261d2219.tiff
  04261d3614.tiff
  04261d3659.tiff
  04261d3666.tiff
  04261d5767.tiff
  04261d6011.tiff
  04261d6013.tiff
  04261d6212.tiff
  05812d1158.tiff
  05842d1039.tiff
  05842d1121.tiff
  05842d1123.tiff
  05856d1892.tiff
  05856d1897.tiff
  05871d936.tiff
  05871d937.tiff
  05963d1103.tiff
  06000d1570.tiff
  06000d1616.tiff
  06000d1714.tiff
  06000d1947.tiff
  06000d2386.tiff
  06000d2741.tiff
  06017d2834.tiff
  06017d3105.tiff
  06017d3636.tiff
  06017d3907.ti

In [6]:

from pathlib import Path

ROOT = Path("/kaggle/input")
terms = (".csv", ".txt", ".tsv", ".xlsx", ".json", ".mat")
matches = []

for p in ROOT.rglob("*"):
    if p.is_file() and p.suffix.lower() in terms:
        matches.append(p)

print(f"Annotation/document files found: {len(matches)}")
for p in matches[:250]:
    print(p)

print("\nKnown example filenames from the published NDCLD literature:")
for name, documented_class in [
    ("04233d2567.tiff", "No Lens"),
    ("04261d994.tiff", "Soft Lens"),
    ("07015d1314.tiff", "Textured Lens"),
]:
    print(f"{name} -> {documented_class} (published example only)")


Annotation/document files found: 0

Known example filenames from the published NDCLD literature:
04233d2567.tiff -> No Lens (published example only)
04261d994.tiff -> Soft Lens (published example only)
07015d1314.tiff -> Textured Lens (published example only)


In [7]:

from pathlib import Path

ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working/nd_iris_project"),
]

for root in ROOTS:
    print(f"\n{'='*60}\n{root}\n{'='*60}")
    if not root.exists():
        print("Not found")
        continue

    matches = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in {
            ".csv", ".txt", ".tsv", ".xlsx", ".json", ".mat", ".xml"
        }:
            matches.append(p)

    print(f"Metadata/annotation candidates: {len(matches)}")
    for p in matches[:100]:
        print(p)



/kaggle/input
Metadata/annotation candidates: 0

/kaggle/working/nd_iris_project
Metadata/annotation candidates: 16
/kaggle/working/nd_iris_project/notebook02_preprocessing/ND_Iris_preprocessing_summary.json
/kaggle/working/nd_iris_project/notebook02_preprocessing/ND_Iris_output_verification_errors.csv
/kaggle/working/nd_iris_project/notebook02_preprocessing/ND_Iris_preprocessed_metadata.csv
/kaggle/working/nd_iris_project/notebook02_preprocessing/ND_Iris_preprocessing_errors.csv
/kaggle/working/nd_iris_project/notebook01_dataset_analysis/ND_Iris_dataset_summary.json
/kaggle/working/nd_iris_project/notebook01_dataset_analysis/ND_Iris_image_dimensions.csv
/kaggle/working/nd_iris_project/notebook01_dataset_analysis/ND_Iris_file_metadata.csv
/kaggle/working/nd_iris_project/notebook01_dataset_analysis/ND_Iris_unreadable_files.csv
/kaggle/working/nd_iris_project/notebook01_dataset_analysis/ND_Iris_folder_counts.csv
/kaggle/working/nd_iris_project/notebook04_baseline_recognition/ND_Iris_ide

In [8]:

# ============================================================
# NOTEBOOK 04 — EXPLORATORY FILENAME-GROUP CLASSIFICATION
# IMPORTANT: NOT A VERIFIED CONTACT-LENS CLASSIFIER
# ============================================================
from pathlib import Path
from PIL import Image
from collections import Counter
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json, re, time, zipfile, random, os, copy

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

START = time.time()
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT = Path("/kaggle/working/nd_iris_project")
SOURCE = Path("/kaggle/input/datasets/pranavsunilwagh/nd-iris/NDCLD15")
IMAGE_DIR = PROJECT / "notebook02_preprocessing/images_224"
OUT = PROJECT / "notebook04_baseline_recognition"
OUT.mkdir(parents=True, exist_ok=True)

# Use existing preprocessed images; don't silently relabel as lens classes.
if not IMAGE_DIR.exists():
    raise FileNotFoundError(f"Preprocessed images missing: {IMAGE_DIR}")

meta_path = PROJECT / "notebook02_preprocessing/ND_Iris_preprocessed_metadata.csv"
meta = pd.read_csv(meta_path)
meta["processed_path"] = meta["processed_path"].astype(str)
meta = meta[meta["processed_path"].map(lambda x: Path(x).is_file())].copy()

# Proxy target: leading filename token before 'd', e.g. 04261d2219.tiff -> 04261
# This is an unverified filename group, NOT a verified identity or lens class.
meta["filename"] = meta["filename"].astype(str)
meta["proxy_label"] = meta["filename"].str.extract(r"^(\d+)d", expand=False)
meta = meta.dropna(subset=["proxy_label"]).reset_index(drop=True)
if len(meta) < 30:
    raise RuntimeError("Too few images have the expected filename pattern.")

encoder = LabelEncoder()
meta["target"] = encoder.fit_transform(meta["proxy_label"])
class_counts = meta["proxy_label"].value_counts()
meta.to_csv(OUT / "ND_Iris_exploratory_labeled_manifest.csv", index=False)

# Random image-level split. This is exploratory only and may share filename
# groups across splits; it does NOT evaluate unseen-subject generalization.
indices = np.arange(len(meta))
train_idx, temp_idx = train_test_split(
    indices, test_size=0.30, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, random_state=SEED
)

train_df = meta.iloc[train_idx].reset_index(drop=True)
val_df = meta.iloc[val_idx].reset_index(drop=True)
test_df = meta.iloc[test_idx].reset_index(drop=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
print(f"Device: {device}; available GPUs: {gpu_count}")
if gpu_count:
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# Image loading is the main pipeline; keep workers modest for Kaggle.
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

class IrisProxyDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        with Image.open(row["processed_path"]) as im:
            x = transform(im.convert("L"))
        return x, int(row["target"])

batch_size = 64
workers = min(4, os.cpu_count() or 2)
train_loader = DataLoader(
    IrisProxyDataset(train_df), batch_size=batch_size, shuffle=True,
    num_workers=workers, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    IrisProxyDataset(val_df), batch_size=batch_size, shuffle=False,
    num_workers=workers, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    IrisProxyDataset(test_df), batch_size=batch_size, shuffle=False,
    num_workers=workers, pin_memory=(device.type == "cuda")
)

num_classes = len(encoder.classes_)
model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(model.fc.in_features, num_classes)
if gpu_count > 1:
    model = nn.DataParallel(model)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 3

def run_epoch(loader, training):
    model.train(training)
    loss_sum, n, correct = 0.0, 0, 0
    all_y, all_pred = [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.backward()
                optimizer.step()
            preds = logits.argmax(1)
            loss_sum += loss.item() * len(yb)
            n += len(yb)
            correct += (preds == yb).sum().item()
            all_y.extend(yb.detach().cpu().tolist())
            all_pred.extend(preds.detach().cpu().tolist())
    return loss_sum / max(n, 1), correct / max(n, 1), all_y, all_pred

history = []
best_val = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc, _, _ = run_epoch(train_loader, True)
    va_loss, va_acc, _, _ = run_epoch(val_loader, False)
    history.append({
        "epoch": epoch, "train_loss": tr_loss, "train_accuracy": tr_acc,
        "val_loss": va_loss, "val_accuracy": va_acc,
        "epoch_seconds": time.time() - t0
    })
    if va_acc > best_val:
        best_val = va_acc
        best_state = copy.deepcopy(model.state_dict())
    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss {tr_loss:.4f}, acc {tr_acc:.4f} | "
        f"val loss {va_loss:.4f}, acc {va_acc:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

if best_state is not None:
    model.load_state_dict(best_state)

test_loss, test_acc, y_true, y_pred = run_epoch(test_loader, False)
history_df = pd.DataFrame(history)
history_df.to_csv(OUT / "ND_Iris_baseline_training_history.csv", index=False)

# Save checkpoint and class mapping. DataParallel prefix is handled at load time.
model_to_save = model.module if isinstance(model, nn.DataParallel) else model
torch.save({
    "model_state_dict": model_to_save.state_dict(),
    "num_classes": num_classes,
    "classes": encoder.classes_.tolist(),
    "input_channels": 1,
    "input_size": [224, 224],
    "target_definition": "unverified filename prefix before 'd'",
    "not_a_contact_lens_label_model": True
}, OUT / "ND_Iris_exploratory_baseline_resnet18.pth")

pred_df = test_df[["filename", "proxy_label", "target"]].copy()
pred_df["true_proxy_class"] = [encoder.classes_[i] for i in y_true]
pred_df["predicted_proxy_class"] = [encoder.classes_[i] for i in y_pred]
pred_df["correct"] = np.asarray(y_true) == np.asarray(y_pred)
pred_df.to_csv(OUT / "ND_Iris_exploratory_test_predictions.csv", index=False)

report = classification_report(
    y_true, y_pred, labels=np.arange(num_classes),
    target_names=encoder.classes_, output_dict=True, zero_division=0
)
with open(OUT / "ND_Iris_exploratory_classification_report.json", "w") as f:
    json.dump(report, f, indent=2)

metrics = {
    "dataset_images_used": int(len(meta)),
    "proxy_class_count": int(num_classes),
    "train_images": int(len(train_df)),
    "validation_images": int(len(val_df)),
    "test_images": int(len(test_df)),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_acc),
    "best_validation_accuracy": float(best_val),
    "epochs": EPOCHS,
    "device": str(device),
    "gpu_count": int(gpu_count),
    "proxy_target": "leading numeric filename token before 'd'",
    "verified_contact_lens_labels_available": False,
    "identity_labels_verified": False,
    "scientific_interpretation": (
        "Exploratory prediction of filename groups only. Not contact-lens "
        "classification, not verified identity recognition, and not evidence "
        "of biometric performance."
    )
}
with open(OUT / "ND_Iris_exploratory_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Training curve plot
plt.figure(figsize=(8, 4))
plt.plot(history_df["epoch"], history_df["train_accuracy"], marker="o", label="Train")
plt.plot(history_df["epoch"], history_df["val_accuracy"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy on filename proxy labels")
plt.title("Exploratory filename-group baseline")
plt.legend()
plt.tight_layout()
plt.savefig(OUT / "ND_Iris_exploratory_baseline_accuracy.png", dpi=140)
plt.close()

conclusion = f"""# Notebook 04 — Exploratory Baseline: Results and Limitations

## Results
- Images used: {len(meta):,}
- Unverified filename-prefix classes: {num_classes:,}
- Training images: {len(train_df):,}
- Validation images: {len(val_df):,}
- Test images: {len(test_df):,}
- Epochs: {EPOCHS}
- Best validation accuracy on proxy labels: {best_val:.4f}
- Test accuracy on proxy labels: {test_acc:.4f}
- Test loss: {test_loss:.4f}
- Device: {device}; GPUs available: {gpu_count}

## Critical limitations
No verified contact-lens class annotations were available. The target was therefore
constructed from the numeric filename prefix before the letter `d`. This is an
unverified proxy, not a No Lens / Soft Lens / Textured Lens label and not a verified
person/eye identity label.

The split is image-level and may contain related filename groups in multiple splits.
Consequently, these metrics describe only prediction of the proxy labels and must not
be presented as contact-lens detection or biometric identity recognition performance.

## Outputs
- Exploratory labeled manifest and class mapping in the checkpoint
- Training history CSV and accuracy plot
- Model checkpoint
- Test predictions CSV and classification report JSON
- Metrics JSON
- This conclusion report

## Conclusion
The notebook completed an exploratory ResNet18 experiment using the available images.
The experiment demonstrates the training pipeline runs, but it does not establish
contact-lens classification performance. Verified annotations and a suitable split
are required for that claim.
"""
(OUT / "Notebook04_Conclusion.md").write_text(conclusion)

# ZIP every result generated by Notebook 04, including model checkpoint.
ZIP_PATH = PROJECT / "ND_Iris_Notebook04_Results.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, arcname=f"notebook04_baseline_recognition/{p.relative_to(OUT)}")

with zipfile.ZipFile(ZIP_PATH) as z:
    bad = z.testzip()
    archived = z.namelist()

print("\n" + "=" * 65)
print("NOTEBOOK 04 — EXPLORATORY BASELINE RESULTS")
print("=" * 65)
print(json.dumps(metrics, indent=2))
print(f"Archived files: {len(archived)}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")
print(f"ZIP integrity: {'PASSED' if bad is None else 'FAILED: ' + str(bad)}")
print(f"Checkpoint: {OUT / 'ND_Iris_exploratory_baseline_resnet18.pth'}")
print(f"Conclusion Markdown: {OUT / 'Notebook04_Conclusion.md'}")
print(f"Results ZIP: {ZIP_PATH}")
print(f"Elapsed: {(time.time()-START)/60:.2f} minutes")
assert bad is None, "ZIP integrity check failed"


Device: cuda; available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4
Epoch 1/3 | train loss 4.8794, acc 0.1623 | val loss 4.4092, acc 0.2075 | 11.3s
Epoch 2/3 | train loss 4.0911, acc 0.2281 | val loss 5.6792, acc 0.0310 | 8.5s
Epoch 3/3 | train loss 3.3401, acc 0.3003 | val loss 3.5404, acc 0.2920 | 8.9s

NOTEBOOK 04 — EXPLORATORY BASELINE RESULTS
{
  "dataset_images_used": 6233,
  "proxy_class_count": 324,
  "train_images": 4363,
  "validation_images": 935,
  "test_images": 935,
  "test_loss": 3.6978408736978623,
  "test_accuracy": 0.2449197860962567,
  "best_validation_accuracy": 0.29197860962566846,
  "epochs": 3,
  "device": "cuda",
  "gpu_count": 2,
  "proxy_target": "leading numeric filename token before 'd'",
  "verified_contact_lens_labels_available": false,
  "identity_labels_verified": false,
  "scientific_interpretation": "Exploratory prediction of filename groups only. Not contact-lens classification, not verified identity recognition, and not evidence of biometric performance.

## Conclusion — Notebook 04: Exploratory Baseline Recognition Model

### Experimental setup

A grayscale ResNet18 model was trained on the 6,233 available NDCLD15 images. Since verified contact-lens annotations were unavailable, the numeric filename prefix before the letter `d` was used as an exploratory proxy target.

- **Total images:** 6,233
- **Proxy classes:** 324
- **Training images:** 4,363
- **Validation images:** 935
- **Test images:** 935
- **Training epochs:** 3
- **Hardware:** 2 NVIDIA Tesla T4 GPUs

### Results

| Metric | Result |
|---|---:|
| Best validation accuracy | 29.20% |
| Test accuracy | 24.49% |
| Test loss | 3.6978 |
| Final training accuracy | 30.03% |

### Interpretation

The model completed three training epochs and achieved a test accuracy of 24.49% when predicting the filename-derived proxy groups. Validation accuracy varied across epochs, so these results should be interpreted as an exploratory baseline rather than evidence of reliable classification.

### Limitations

No verified ground-truth labels for the three contact-lens categories—No Lens, Soft Lens, and Textured Lens—were available in the attached dataset. The 324 proxy classes are derived from filename prefixes and must not be interpreted as contact-lens categories or verified subject identities.

The image-level split may place related filename groups in multiple partitions. Consequently, the reported metrics do not establish generalization to unseen subjects and cannot be reported as contact-lens detection accuracy or biometric recognition performance.

### Generated artifacts

- Exploratory labeled manifest
- ResNet18 model checkpoint
- Training history CSV
- Test predictions CSV
- Classification report JSON
- Metrics JSON
- Training accuracy plot
- `ND_Iris_Notebook04_Results.zip`

**ZIP integrity:** Passed.

### Overall conclusion

Notebook 04 successfully established and executed an exploratory GPU-based ResNet18 training pipeline. The results provide a technical baseline for the available images, but meaningful contact-lens classification requires verified class annotations and an appropriate evaluation split.